# Track A: classical clustering and CVRP, step by step

**S only:** four customers, two vehicles, capacity seven. The algorithms below
are executable notebook code, not calls to the command-line runner.
Customer locations, depot, and demand vectors come from
[inputs/seeds_s.json](inputs/seeds_s.json). No credentials are needed.

Shared imports are limited to validated inputs, persistence, the QUBO comparison
whose derivation is in the quantum notebook, and an installed-solver cross-check.
The clustering objective, partition search, exact routing, references,
annealing, and experiment loop are defined here. Their CLI counterparts remain
in `common/` for command-line reproduction.

## Before the code: what are we solving?

Imagine four customers needing deliveries. Each has a location and a demand
(how much must be delivered). We have two vehicles, each able to carry at most
seven demand units.

There are **two different questions**:
1. **Clustering:** which customers share a vehicle? We prefer groups whose
   customers are near each other. We do not choose a visiting order yet.
2. **CVRP evaluation:** starting at the depot (the warehouse), how far must
   the vehicles travel to serve those groups and return?

A **partition** is simply a division into non-overlapping groups. For example,
labels `(0, 1, 1, 0)` put customers 0 and 3 in one vehicle and customers 1 and
2 in the other. These labels are not a route order. Swapping vehicle names
does not create a new delivery plan.

A **constraint** is a rule that must be obeyed. An equality means "exactly";
an inequality means "at most" or "at least." Here, every customer must belong
to **exactly one** vehicle and every vehicle must carry **at most seven**
units. An unused vehicle is allowed.

You can read the prose, run each cell, and inspect the tables before studying
the functions. Functions define reusable steps; the final execution cell
actually runs them. Shared input validation computes a feasibility witness:
one legal grouping proving the demands can fit. It is not a hint given to
the optimizers.

In [1]:
from pathlib import Path
import json
import sys
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "track_a" / "inputs" / "seeds_s.json").exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from track_a.common.instances import OUTPUT_PATH, Instance, ensure_instances, feasible, validate
from track_a.common.analysis import fingerprint, save
from track_a.common.models import ENCODING

instances = ensure_instances()
print("Processing only the first demand vector:", instances[0].demands)
output = OUTPUT_PATH
shots = 256
seed = 0

from collections import Counter
from dataclasses import asdict
from functools import lru_cache
from itertools import permutations
from math import inf
from track_a.common.models import Qubo, build_qubo, encode, validate_small_qubo
from track_a.common.classical import installed_qubo_crosscheck

display(pd.DataFrame([{"instance": i.id, "demands": i.demands, **i.ratios}
                      for i in instances]))

Processing only the first demand vector: (2, 3, 4, 2)


,instance,demands,rho,largest_demand_ratio
0,s-0,"(2, 3, 4, 2)",0.785714,0.571429


## 1. Define compactness and remove vehicle-label symmetry

For binary assignments, minimize
$F=\sum_k\sum_{i<j}c_{ij}x_{ik}x_{jk}$.
Each customer has exactly one vehicle (**equality**); each load is at most Q
(**inequality**). Vehicles may be unused. `canonical` identifies relabelings
of the same partition. The depot is intentionally absent from F.

**Reading the formula:** i and j identify two
customers, k identifies a vehicle, and c is the straight-line distance between
the customers. An assignment variable x is 1 when that customer uses that
vehicle. The product of two x values is 1 only when both customers use it,
so we add that pair's distance only when they share a vehicle. Smaller totals
mean more compact groups; this total is not a vehicle's driving distance.

In [2]:
def objective(instance: Instance, labels: tuple[int, ...] | list[int]) -> float:
    distances = instance.distances
    return float(sum(distances[i, j] for i in range(instance.n)
                     for j in range(i) if labels[i] == labels[j]))


def canonical(labels: tuple[int, ...] | list[int]) -> tuple[int, ...]:
    mapping: dict[int, int] = {}
    return tuple(mapping.setdefault(k, len(mapping)) for k in labels)

## 2. Enumerate each candidate partition exactly once

Restricted-growth labels give eight unlabeled S partitions before
capacity filtering. This search does not use the feasibility witness as a
warm start or fix its groups.

**What the code is doing:** it tries the
different ways to split four customers between up to two vehicles. It gives
the first group the name 0, and introduces group 1 only when needed. This
avoids counting the same groups again just because vehicle names changed.

In [3]:
def partitions(n: int, vehicles: int):
    """Restricted-growth strings: each unlabeled partition exactly once."""
    labels = [0] * n

    def visit(i: int, maximum: int):
        if i == n:
            yield tuple(labels)
            return
        for k in range(min(maximum + 1, vehicles - 1) + 1):
            labels[i] = k
            yield from visit(i + 1, max(maximum, k))

    yield from visit(1, 0)

## 3. Route fixed groups exactly, without reassignment

For each group, Held-Karp considers all visit orders and closes the
tour through the depot. The cache is independent of demands. `evaluate`
checks coverage and capacity and never changes a group's membership.
Empty groups cost zero.

**Why this is a separate step:** knowing who
shares a vehicle does not say whom to visit first. The depot is the start and
finish. Held-Karp is an exact bookkeeping method that builds short paths from
smaller ones instead of recalculating every repeated path. "Exact" means no
possible better visit order is missed. A cache remembers an already computed
tour so another method can reuse the same fair evaluation.

In [4]:
class Routing:
    """Exact tours shared across demand permutations with the same geometry."""

    def __init__(self, instance: Instance):
        self.coordinates = instance.coordinates
        self.depot = instance.depot
        self.distances = instance.distances
        self.to_depot = np.linalg.norm(np.asarray(instance.coordinates) - instance.depot, axis=1)
        self.tours: dict[tuple[int, ...], tuple[float, tuple[int, ...]]] = {(): (0.0, ())}

    def tour(self, group: tuple[int, ...]) -> tuple[float, tuple[int, ...]]:
        group = tuple(sorted(group))
        if group in self.tours:
            return self.tours[group]

        @lru_cache(None)
        def path(mask: int, last: int) -> tuple[float, tuple[int, ...]]:
            previous = mask ^ (1 << last)
            customer = group[last]
            if previous == 0:
                return float(self.to_depot[customer]), (customer,)
            cost, order = min(
                (path(previous, j)[0] + self.distances[group[j], customer],
                 path(previous, j)[1])
                for j in range(len(group)) if previous & (1 << j)
            )
            return float(cost), order + (customer,)

        mask = (1 << len(group)) - 1
        cost, order = min(
            (path(mask, last)[0] + self.to_depot[group[last]], path(mask, last)[1])
            for last in range(len(group))
        )
        result = float(cost), order
        self.tours[group] = result
        return result

    def evaluate(self, instance: Instance, labels: tuple[int, ...]) -> dict:
        if (instance.coordinates != self.coordinates or instance.depot != self.depot
                or not feasible(instance, labels)):
            raise ValueError("Invalid fixed-group routing request")
        tours, cost = [], 0.0
        covered = []
        for k in range(instance.vehicles):
            group = tuple(i for i, v in enumerate(labels) if v == k)
            length, order = self.tour(group)
            covered.extend(order)
            if set(order) != set(group):
                raise AssertionError("Routing changed customer groups")
            tours.append({"vehicle": k, "customers": list(order),
                          "route": ["depot", *order, "depot"], "cost": length,
                          "load": sum(instance.demands[i] for i in group)})
            cost += length
        if sorted(covered) != list(range(instance.n)):
            raise AssertionError("Route coverage failure")
        return {"cost": cost, "tours": tours}

## 4. Compute the cost of every possible group

The subset table records compactness and exact depot-tour costs
separately. Full CVRP can choose groups using tour cost; clustering chooses
them using compactness. They are different optimization problems.

**Subsets in everyday terms:** a subset is any
selection of customers, such as customers 0 and 2. We calculate both scores
for each selection. Later, the optimizer combines selections that cover all
customers without overlap. The integer masks in this code are compact ways
of recording which customers are selected, not additional physical qubits.

In [5]:
@lru_cache(maxsize=3)
def subset_costs(coordinates: tuple[tuple[float, float], ...],
                 depot: tuple[float, float]):
    """All group clustering costs and exact depot tour costs, independent of demand."""
    n = len(coordinates)
    distances = np.linalg.norm(np.asarray(coordinates)[:, None] - np.asarray(coordinates)[None, :], axis=-1)
    to_depot = np.linalg.norm(np.asarray(coordinates) - depot, axis=1)
    compactness = np.zeros(1 << n)
    paths = np.full((1 << n, n), np.inf)
    tours = np.zeros(1 << n)
    for mask in range(1, 1 << n):
        bit = mask & -mask
        i = bit.bit_length() - 1
        previous = mask ^ bit
        compactness[mask] = compactness[previous] + sum(
            distances[i, j] for j in range(n) if previous & (1 << j))
        members = [j for j in range(n) if mask & (1 << j)]
        for last in members:
            rest = mask ^ (1 << last)
            paths[mask, last] = (to_depot[last] if rest == 0 else min(
                paths[rest, j] + distances[j, last] for j in members if j != last))
        tours[mask] = min(paths[mask, j] + to_depot[j] for j in members)
    return compactness, tours

## 5. Independently cross-check both constrained objectives with CPLEX

Choose capacity-feasible group columns, cover every customer exactly
once, and use at most K groups. This set-partition MILP is equivalent to the
original CVRP formulation when group costs are exact tours. The complete arc,
flow, depot, capacity, and subtour derivation is in [MODEL.md](docs/MODEL.md).

**Why another solver?** CPLEX is a
classical optimization program. Asking it to solve the same constrained
problem checks our own implementation. Each "column" here means one possible
customer group; a binary choice says whether to use that group. A MILP
(mixed-integer linear program) is a model with linear costs/rules and discrete
decisions. A time-limit or solver error is kept visible rather than treated
as proof of success.

In [6]:
def cplex_crosscheck(instance: Instance, reference: dict, time_limit: float = 30.0) -> dict:
    """Independent set-partition MILPs on S, with exact group-tour columns."""
    from docplex.mp.model import Model
    from docplex.mp.utils import DOcplexException

    start = perf_counter()
    compactness, tours = subset_costs(instance.coordinates, instance.depot)
    groups = [mask for mask in range(1, 1 << instance.n) if sum(
        instance.demands[i] for i in range(instance.n) if mask & (1 << i)
    ) <= instance.capacity]
    outputs = {}
    for name, costs in (("clustering", compactness), ("cvrp", tours)):
        try:
            with Model(name=f"track_a_{name}") as model:
                variables = model.binary_var_list(len(groups))
                for i in range(instance.n):
                    model.add_constraint(model.sum(variables[j] for j, mask in enumerate(groups)
                                                   if mask & (1 << i)) == 1)
                model.add_constraint(model.sum(variables) <= instance.vehicles)
                model.minimize(model.sum(float(costs[mask]) * variables[j]
                                         for j, mask in enumerate(groups)))
                model.parameters.timelimit = time_limit
                model.parameters.threads = 1
                model.parameters.randomseed = 0
                solution = model.solve(log_output=False)
                if solution is None:
                    outputs[name] = {"status": model.solve_details.status, "cost": None}
                else:
                    value = solution.objective_value
                    if abs(value - reference[name]["cost"]) > 1e-6:
                        raise AssertionError("CPLEX and exhaustive reference disagree")
                    outputs[name] = {"status": model.solve_details.status, "cost": value,
                                     "agrees": True}
        except DOcplexException as error:
            outputs[name] = {"status": "solver_error", "error": str(error)}
    return {"settings": {"time_limit_seconds": time_limit, "threads": 1, "seed": 0},
            "seconds": perf_counter() - start,
            **outputs}

## 6. Find both exact references and count S optima

The recursion anchors each next group on the least remaining customer,
so vehicle relabelings are not counted twice. It keeps separate best costs,
counts, and reconstruction choices for clustering and full CVRP.
The final S check counts route optima modulo reversal.

**Reading this search:** imagine choosing one group,
then solving the smaller problem for customers still unassigned. The recursion
repeats that idea until everyone is assigned. It records which choices gave
the best score so we can reconstruct the answer, not just report a number.
A tour and the same tour driven backwards have the same length here.

In [7]:
def exact_references(instance: Instance) -> dict:
    start = perf_counter()
    compactness, tour_cost = subset_costs(instance.coordinates, instance.depot)
    loads = np.zeros(1 << instance.n, dtype=int)
    for mask in range(1, len(loads)):
        bit = mask & -mask
        loads[mask] = loads[mask ^ bit] + instance.demands[bit.bit_length() - 1]

    @lru_cache(None)
    def solve(mask: int, vehicles: int) -> tuple[float, float, int, int, int, int]:
        if mask == 0:
            return 0.0, 0.0, 1, 1, 0, 0
        if vehicles == 0 or loads[mask] > vehicles * instance.capacity:
            return inf, inf, 0, 0, 0, 0
        if vehicles == 1:
            if loads[mask] <= instance.capacity:
                return float(compactness[mask]), float(tour_cost[mask]), 1, 1, mask, mask
            return inf, inf, 0, 0, 0, 0
        first = mask & -mask
        sub = mask
        best_f = best_c = inf
        count_f = count_c = choice_f = choice_c = 0
        while sub:
            if sub & first and loads[sub] <= instance.capacity:
                f, c, nf, nc, _, _ = solve(mask ^ sub, vehicles - 1)
                f += compactness[sub]
                c += tour_cost[sub]
                if nf:
                    if f < best_f - 1e-8:
                        best_f, count_f, choice_f = float(f), nf, sub
                    elif abs(f - best_f) <= 1e-8:
                        count_f += nf
                if nc:
                    if c < best_c - 1e-8:
                        best_c, count_c, choice_c = float(c), nc, sub
                    elif abs(c - best_c) <= 1e-8:
                        count_c += nc
            sub = (sub - 1) & mask
        return best_f, best_c, count_f, count_c, choice_f, choice_c

    full = (1 << instance.n) - 1
    f, c, nf, nc, _, _ = solve(full, instance.vehicles)

    def assignment(which: int) -> tuple[int, ...]:
        labels = [0] * instance.n
        mask, remaining, vehicle = full, instance.vehicles, 0
        while mask:
            sub = solve(mask, remaining)[which]
            if not sub:
                raise AssertionError("Exact reference reconstruction failed")
            for i in range(instance.n):
                if sub & (1 << i):
                    labels[i] = vehicle
            mask ^= sub
            remaining -= 1
            vehicle += 1
        return canonical(labels)

    result = {
        "clustering": {"cost": f, "labels": assignment(4), "distinct_optima": nf,
                       "status": "proven_optimum"},
        "cvrp": {"cost": c, "labels": assignment(5), "optimal_partitions": nc,
                 "status": "proven_optimum"},
        "status": "proven_optimum", "algorithm": "exhaustive-subset-dynamic-programming",
        "settings": {"maximum_customers": 15, "vehicle_labels_quotiented": True,
                     "equality_tolerance": 1e-8},
        "seconds": perf_counter() - start,
    }
    if instance.tier == "s":
        routing = Routing(instance)
        route_optima = 0
        feasible_partitions = 0
        for labels in partitions(instance.n, instance.vehicles):
            if not feasible(instance, labels):
                continue
            feasible_partitions += 1
            if abs(routing.evaluate(instance, labels)["cost"] - c) > 1e-8:
                continue
            multiplicity = 1
            for k in range(instance.vehicles):
                group = tuple(i for i, v in enumerate(labels) if v == k)
                cost, _ = routing.tour(group)
                orders = set()
                for order in permutations(group):
                    length = (0.0 if not order else routing.to_depot[order[0]] +
                              sum(routing.distances[a, b] for a, b in zip(order, order[1:])) +
                              routing.to_depot[order[-1]])
                    if abs(length - cost) <= 1e-8:
                        orders.add(min(order, order[::-1]))
                multiplicity *= len(orders)
            route_optima += multiplicity
        result["cvrp"]["distinct_optima_modulo_reversal"] = route_optima
        result["feasible_partitions"] = feasible_partitions
        result["cplex_crosscheck"] = cplex_crosscheck(instance, result)
    return result

## 7. Sample the same target QUBO classically, without repair

Each Metropolis chain starts from independent random bits. A flip
changes energy by `(1 - 2*z_i) * field_i`. Temperature falls geometrically
over 400 steps. There is no certificate warm start or local repair.

**Simulated annealing is classical:** it
tries changing bits at random. Early on, a high "temperature" sometimes
allows worse moves to escape a poor starting point; later, a low temperature
mostly accepts improvements. This temperature is an algorithm setting, not
the atoms' laboratory temperature. A chain is one run of those repeated
trial changes.

In [8]:
def anneal(qubo: Qubo, shots: int, seed: int, steps: int = 400) -> Counter[str]:
    """Independent Metropolis chains, no repair or certificate warm start."""
    rng = np.random.default_rng(seed)
    size = len(qubo.variable_order)
    z = rng.integers(0, 2, size=(shots, size))
    diagonal = np.diag(qubo.matrix)
    pairs = 2 * (qubo.matrix - np.diag(diagonal))
    fields = z @ pairs + diagonal
    temperatures = np.geomspace(10 * qubo.penalty, 0.01 * qubo.penalty, steps)
    rows = np.arange(shots)
    for temperature in temperatures:
        columns = rng.integers(size, size=shots)
        change = 1 - 2 * z[rows, columns]
        delta = change * fields[rows, columns]
        accepted = (delta <= 0) | (np.log(rng.random(shots)) < -delta / temperature)
        selected_rows, selected_cols = rows[accepted], columns[accepted]
        fields[selected_rows] += change[accepted, None] * pairs[selected_cols]
        z[selected_rows, selected_cols] = 1 - z[selected_rows, selected_cols]
    return Counter("".join(map(str, row)) for row in z)

## 8. Run and save the classical comparisons

This visible loop computes the references, checks every 14-bit target,
and saves constrained, QUBO-SA, and uniform-random results. The constrained
solution is deterministic and repeated to the common shot budget; those copies
are not independent stochastic successes. The QUBO construction and decoding
are written out in the quantum notebook.

**How to read the output:** "constrained" searches only
legal customer groups. "QUBO-SA" searches bitstrings with penalties for broken
rules. "Random bits" does no optimization and is a sanity-check baseline.
All are scored with the same original delivery rules.

In [9]:
def classical_runs(instances, output: Path, shots: int, seed: int) -> None:
    for instance in instances:
        print(f"Classical {instance.id}", flush=True)
        record = asdict(instance)
        reference = exact_references(instance)
        reference["instance_hash"] = fingerprint(record)
        save(output / "references" / f"{instance.id}.json", reference)
        qubo = build_qubo(instance)
        if instance.tier == "s":
            validation = validate_small_qubo(instance, reference["clustering"]["cost"])
            validation["installed_qubo_solver"] = installed_qubo_crosscheck(
                instance, qubo, reference["clustering"]["cost"])
            save(output / "validation" / f"{instance.id}.json",
                 validation)
        config = {"shots": shots, "seed": seed, "annealing_steps": 400,
                  "initial_temperature": 10 * qubo.penalty,
                  "final_temperature": 0.01 * qubo.penalty,
                  "cooling": "geometric", "restarts": shots,
                  "warm_start": False, "repair": False, "penalty": qubo.penalty,
                  "time_limit_seconds": None}
        for method in ("classical-constrained", "classical-qubo-sa", "random-bits"):
            start = perf_counter()
            if method == "classical-constrained":
                labels = min(
                    (labels for labels in partitions(instance.n, instance.vehicles)
                     if feasible(instance, labels)),
                    key=lambda labels: objective(instance, labels),
                )
                counts = {encode(instance, labels): shots}
                seconds = perf_counter() - start
            elif method == "classical-qubo-sa":
                counts = dict(anneal(qubo, shots, seed))
                seconds = perf_counter() - start
            else:
                rng = np.random.default_rng(seed)
                bits = rng.integers(0, 2, (shots, len(qubo.variable_order)))
                counts = dict(Counter("".join(map(str, row)) for row in bits))
                seconds = perf_counter() - start
            save(output / "runs" / f"{instance.id}-{method}-{seed}.json", {
                "instance": record, "instance_id": instance.id, "instance_hash": fingerprint(record),
                "method": method, "seed": seed, "encoding": ENCODING,
                "configuration_id": fingerprint({**config, "method": method}),
                "configuration": {**config, "method": method},
                "status": "completed", "counts": counts, "shot_budget": shots,
                "atom_count": len(qubo.variable_order), "clustering_seconds": seconds,
                "deterministic_solver_result_repeated": method == "classical-constrained",
                "variable_order": qubo.variable_order,
            })

In [10]:
instance_path = output / "instances.json"
if instance_path.exists():
    previous = json.loads(instance_path.read_text())
    current = [{**asdict(i), **i.ratios, "certificate_valid": True} for i in instances]
    if fingerprint(previous) != fingerprint(current):
        raise ValueError("Use a new output directory for changed input data")
save(instance_path, [{**asdict(i), **i.ratios, "certificate_valid": True} for i in instances])
classical_runs(instances, output, shots, seed)
references = {i.id: json.loads((output / "references" / f"{i.id}.json").read_text())
              for i in instances}
display(pd.DataFrame([
    {"instance": name, "model_reference": ref["clustering"]["cost"],
     "cvrp_reference": ref["cvrp"]["cost"], "status": ref["status"],
     "model_optima": ref["clustering"]["distinct_optima"],
     "cvrp_optima": ref["cvrp"]["distinct_optima_modulo_reversal"]}
    for name, ref in references.items()
]))

Classical s-0


,instance,model_reference,cvrp_reference,status,model_optima,cvrp_optima
0,s-0,80.886853,205.64738,proven_optimum,1,1


Continue with the quantum notebook to inspect the binary-slack
expansion, physical approximation and actual pulse. The results notebook
evaluates saved counts without rerunning either optimizer.